# Notebook 05 — Circuit Verification

Verify that all three circuit variants (ideal, unitary, dynamic)
produce the correct magnetisation by running them through a noiseless
simulator and comparing against the exact reference.

**Run this before submitting to hardware.**

Optionally add a noise model to preview how hardware noise affects
the observable.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt

from qiskit_aer import AerSimulator
from qiskit.transpiler import CouplingMap

from src.model import coupling_pairs, exact_trotter_magnetization
from src.circuits import (
    build_ideal_trotter_circuit,
    build_dynamic_trotter_circuit,
    build_dynamic_trotter_circuit_physical,
    transpile_to_ideal,
    transpile_to_heavyhex,
    transpile_dynamic_physical,
)

## Parameters

In [ ]:
N_QUBITS = 6
ALPHAS = [0.5, 2.0, 8.0]
CUTOFF_FRAC = 0.10
THETA = np.pi / 4
STEPS_TO_CHECK = [0, 1, 4, 8]
# For thorough verification, expand to: [0, 1, 2, 4]
SHOTS = 10000                        # 10k shots: ~0.01 shot noise, sufficient for verification
SEED = 42

# Unitary verification: heavy_hex(3) = 19 qubits, fast simulation.
# The transpiler routes SWAPs without needing specific node degrees.
heavy_cmap_unitary = CouplingMap.from_heavy_hex(3, bidirectional=True)

# Dynamic verification: heavy_hex(5) = 57 qubits, required because
# build_dynamic_trotter_circuit_physical places data qubits on degree-3
# nodes with ancillas on degree-2 bridges.  heavy_hex(3) has only 4
# degree-3 nodes — not enough to embed a 6-qubit chain.  heavy_hex(5)
# has 16 degree-3 nodes with chains of length up to 14.
heavy_cmap_dynamic = CouplingMap.from_heavy_hex(5, bidirectional=True)

## Helper: compute M_z from shot counts

In [ ]:
def mz_from_counts(counts, n_data, register_name=None):
    """Compute average magnetisation from measurement counts.

    Parameters
    ----------
    counts : dict
        Qiskit counts dictionary.  Keys may be multi-register
        (space-separated) for dynamic circuits.
    n_data : int
        Number of data qubits.
    register_name : str or None
        If the circuit has multiple classical registers, specify
        which one contains the final measurement (e.g. 'meas').
        If None, uses all bits.

    Returns
    -------
    float : estimated <M_z>.
    """
    total = sum(counts.values())
    mz = 0.0

    for bitstring, count in counts.items():
        parts = bitstring.split()

        if len(parts) > 1:
            # Multi-register: 'meas' is the last register added,
            # so it appears first (leftmost) in Qiskit's output.
            meas_str = parts[0]
        else:
            meas_str = bitstring

        # Qiskit bitstrings: rightmost bit = qubit 0
        bits = [int(b) for b in reversed(meas_str)]
        zi_sum = sum(1 - 2 * b for b in bits[:n_data])
        mz += (zi_sum / n_data) * count

    return mz / total

## 1. Noiseless verification

Run all three variants through a noiseless Aer simulator and
compare against the exact reference.  Differences should be
purely from shot noise (~1/√shots ≈ 0.01 for 10k shots).

In [ ]:
sim = AerSimulator()
results = []

print(f"{'variant':>10} {'alpha':>5} {'steps':>5} | {'Mz_sim':>9} {'Mz_exact':>9} {'diff':>8} {'status':>6}")
print("-" * 65)

for alpha in ALPHAS:
    for n_steps in STEPS_TO_CHECK:
        mz_exact = exact_trotter_magnetization(
            N_QUBITS, alpha, 1.0, THETA, n_steps, cutoff_frac=CUTOFF_FRAC,
        )

        # --- Ideal (logical, no transpilation) ---
        circ_i = build_ideal_trotter_circuit(
            N_QUBITS, alpha, THETA, n_steps, measure=True, cutoff_frac=CUTOFF_FRAC,
        )
        job = sim.run(circ_i, shots=SHOTS)
        mz_i = mz_from_counts(job.result().get_counts(), N_QUBITS)
        diff_i = abs(mz_i - mz_exact)
        ok_i = "✓" if diff_i < 3 / np.sqrt(SHOTS) else "✗"
        results.append(("ideal", alpha, n_steps, mz_i, mz_exact, diff_i))
        print(f"{'ideal':>10} {alpha:>5.1f} {n_steps:>5} | {mz_i:>+9.5f} {mz_exact:>+9.5f} {diff_i:>8.5f} {ok_i:>6}")

        # --- Unitary (transpiled to heavy-hex) ---
        circ_u = transpile_to_heavyhex(circ_i, heavy_cmap_unitary, seed=SEED)
        job = sim.run(circ_u, shots=SHOTS)
        mz_u = mz_from_counts(job.result().get_counts(), N_QUBITS)
        diff_u = abs(mz_u - mz_exact)
        ok_u = "✓" if diff_u < 3 / np.sqrt(SHOTS) else "✗"
        results.append(("unitary", alpha, n_steps, mz_u, mz_exact, diff_u))
        print(f"{'unitary':>10} {alpha:>5.1f} {n_steps:>5} | {mz_u:>+9.5f} {mz_exact:>+9.5f} {diff_u:>8.5f} {ok_u:>6}")

        # --- Dynamic (built on physical topology, transpiled) ---
        circ_d_phys = build_dynamic_trotter_circuit_physical(
            N_QUBITS, alpha, THETA, coupling_map=heavy_cmap_dynamic,
            n_steps=n_steps, measure=True, cutoff_frac=CUTOFF_FRAC,
        )
        circ_d_t = transpile_dynamic_physical(circ_d_phys, heavy_cmap_dynamic, seed=SEED)
        job = sim.run(circ_d_t, shots=SHOTS)
        mz_d = mz_from_counts(job.result().get_counts(), N_QUBITS)
        diff_d = abs(mz_d - mz_exact)
        ok_d = "✓" if diff_d < 3 / np.sqrt(SHOTS) else "✗"
        results.append(("dynamic", alpha, n_steps, mz_d, mz_exact, diff_d))
        print(f"{'dynamic':>10} {alpha:>5.1f} {n_steps:>5} | {mz_d:>+9.5f} {mz_exact:>+9.5f} {diff_d:>8.5f} {ok_d:>6}")

        print()

## 2. Summary: are all circuits correct?

In [ ]:
shot_noise = 1.0 / np.sqrt(SHOTS)
n_fail = sum(1 for r in results if r[5] > 3 * shot_noise)

if n_fail == 0:
    print(f"All {len(results)} checks passed (within 3σ shot noise = {3*shot_noise:.4f})")
else:
    print(f"WARNING: {n_fail} / {len(results)} checks exceeded 3σ threshold")
    for r in results:
        if r[5] > 3 * shot_noise:
            print(f"  {r[0]} α={r[1]} s={r[2]}: diff={r[5]:.5f}")

## 3. Visualise: simulated vs exact

In [ ]:
fig, axes = plt.subplots(1, len(ALPHAS), figsize=(4 * len(ALPHAS), 4), sharey=True)
if len(ALPHAS) == 1:
    axes = [axes]

variant_styles = {
    "ideal":   {"color": "#2E7D32", "marker": "o"},
    "unitary": {"color": "#C62828", "marker": "s"},
    "dynamic": {"color": "#1565C0", "marker": "^"},
}

for ax, alpha in zip(axes, ALPHAS):
    # Exact reference
    all_steps = np.arange(max(STEPS_TO_CHECK) + 1)
    exact_vals = [exact_trotter_magnetization(N_QUBITS, alpha, 1.0, THETA, s,
                  cutoff_frac=1e-14) for s in all_steps]
    ax.plot(all_steps, exact_vals, "k-", lw=2, alpha=0.3, label="exact")

    for variant, style in variant_styles.items():
        sub = [(r[2], r[3]) for r in results if r[0] == variant and r[1] == alpha]
        if sub:
            ss, mzs = zip(*sub)
            ax.plot(ss, mzs, linestyle="none", markersize=8,
                    label=variant, **style)

    ax.set_title(rf"$\alpha = {alpha}$")
    ax.set_xlabel("Trotter steps")
    ax.grid(True, alpha=0.2)

axes[0].set_ylabel(r"$\langle M_z \rangle$")
axes[0].legend(fontsize=8)
fig.suptitle(f"Circuit verification (N={N_QUBITS}, {SHOTS} shots, noiseless)", fontsize=13)
fig.tight_layout()
plt.savefig("../results/verification_noiseless.png", dpi=150)
plt.show()

## 4. Optional: noisy simulation

Uncomment the cell below to add a simple depolarising noise model
and see how hardware-like noise affects the observable.
This previews what you'll see on real hardware.

In [ ]:
# === UNCOMMENT THIS CELL TO RUN NOISY SIMULATION ===
#
# from qiskit_aer.noise import NoiseModel, depolarizing_error
#
# def make_noise_model(cx_error=0.01, meas_error=0.02):
#     """Simple depolarising noise model."""
#     noise = NoiseModel()
#     noise.add_all_qubit_quantum_error(
#         depolarizing_error(cx_error, 2), ["cx", "ecr", "rzz"]
#     )
#     noise.add_all_qubit_quantum_error(
#         depolarizing_error(meas_error, 1), ["measure"]
#     )
#     return noise
#
# noise = make_noise_model(cx_error=0.005, meas_error=0.01)
# sim_noisy = AerSimulator(noise_model=noise)
#
# print("=== Noisy simulation (CX error=0.5%, meas error=1%) ===")
# print(f"{'variant':>10} {'alpha':>5} {'steps':>5} | {'Mz_noisy':>9} {'Mz_exact':>9} {'error':>8}")
# print("-" * 55)
#
# for alpha in [1.0, 2.0, 4.0]:
#     for n_steps in [1, 4]:
#         mz_exact = exact_trotter_magnetization(
#             N_QUBITS, alpha, 1.0, THETA, n_steps, cutoff_frac=CUTOFF_FRAC,
#         )
#
#         # Unitary
#         circ = build_ideal_trotter_circuit(
#             N_QUBITS, alpha, THETA, n_steps, measure=True, cutoff_frac=CUTOFF_FRAC,
#         )
#         circ_t = transpile_to_heavyhex(circ, heavy_cmap_unitary, seed=SEED)
#         job = sim_noisy.run(circ_t, shots=SHOTS)
#         mz = mz_from_counts(job.result().get_counts(), N_QUBITS)
#         print(f"{'unitary':>10} {alpha:>5.1f} {n_steps:>5} | {mz:>+9.5f} {mz_exact:>+9.5f} {abs(mz-mz_exact):>8.5f}")
#
#         # Dynamic (physical)
#         circ_d = build_dynamic_trotter_circuit_physical(
#             N_QUBITS, alpha, THETA, coupling_map=heavy_cmap_dynamic,
#             n_steps=n_steps, measure=True, cutoff_frac=CUTOFF_FRAC,
#         )
#         circ_d_t = transpile_dynamic_physical(circ_d, heavy_cmap_dynamic, seed=SEED)
#         job = sim_noisy.run(circ_d_t, shots=SHOTS)
#         mz = mz_from_counts(job.result().get_counts(), N_QUBITS)
#         print(f"{'dynamic':>10} {alpha:>5.1f} {n_steps:>5} | {mz:>+9.5f} {mz_exact:>+9.5f} {abs(mz-mz_exact):>8.5f}")
#     print()